In [ ]:
import pandas as pd
import lightgbm as lgb
import time
import joblib
import gc
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

filenames = {
    "Original":         "original.csv",
    "Small GAN":        "small_GAN_combined.csv",
    "Med GAN":          "med_GAN_combined.csv",
    "Large GAN":        "large_GAN_combined.csv",
    "Small Diffusion":  "small_Diffusion_combined.csv",
    "Med Diffusion":    "med_Diffusion_combined.csv",
    "Large Diffusion":  "large_Diffusion_combined.csv",
    "Small LLM":        "small_LLM_combined.csv",
    "Med LLM":          "med_LLM_combined.csv",
    # "Large LLM":        "large_LLM_combined.csv",
}

df_orig = pd.read_csv(filenames["Original"])
X_all_orig = df_orig.drop('class1', axis=1)
y_all_orig = df_orig['class1']

X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_all_orig, y_all_orig, test_size=0.2, random_state=42
)


num_classes = y_all_orig.nunique()
# joblib.dump(le, 'label_encoder.joblib') # for saving le in original jupyter
le = joblib.load('label_encoder.joblib')
results = []
dist_data = []

for name, path in filenames.items():
    print(f"Processing {name}...")
    start_time = time.time()

    if name == "Original":
        X_train, y_train = X_train_orig, y_train_orig
    else:
        df_temp = pd.read_csv(path)
        X_train = df_temp.drop('class1', axis=1)
        y_train = df_temp['class1']
        del df_temp

    counts = y_train.value_counts(normalize=True)
    for label_idx, ratio in counts.items():
        # Map the integer back to the string name
        class_name = le.inverse_transform([int(label_idx)])[0]

        dist_data.append({
            "Dataset": name,
            "Class": class_name,
            "Percentage": ratio * 100
        })

    model = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=num_classes,
        device='gpu',
        gpu_platform_id=0,
        gpu_device_id=0,
        n_estimators=200,
        learning_rate=0.05,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test_orig)
    y_prob = model.predict_proba(X_test_orig)

    results.append({
        "Dataset": name,
        "Time (s)": round(time.time() - start_time, 2),
        "Accuracy": round(accuracy_score(y_test_orig, y_pred), 4),
        "F1 (Macro)": round(f1_score(y_test_orig, y_pred, average='macro'), 4),
        "AUC (OvR)": round(roc_auc_score(y_test_orig, y_prob, multi_class='ovr'), 4)
    })

    if name == "Original":
        del X_train, y_train
        del X_train_orig, y_train_orig
    else:
        del X_train, y_train

    gc.collect()

In [ ]:
import plotly.express as px

df_res = pd.DataFrame(results)

orig_metrics = df_res[df_res['Method'] == 'Original'].iloc[0]
df_plot = df_res[df_res['Method'] != 'Original'].copy()

df_melted = df_plot.melt(id_vars=['Method', 'Size'],
                         value_vars=['Accuracy', 'F1 (Macro)', 'AUC (OvR)'],
                         var_name='Metric', value_name='Score')

fig = px.bar(
    df_melted,
    x="Method",
    y="Score",
    color="Size",
    barmode="group",
    facet_col="Metric",
    category_orders={"Size": ["Small", "Med", "Large"],
                     "Method": ["GAN", "Diffusion", "LLM"]},
    title="Synthetic Data Utility: Comparison of Generation Methods and Sample Sizes",
    labels={"Score": "Performance Score (on Real Test Set)"},
    height=500
)

for i, metric in enumerate(['Accuracy', 'F1 (Macro)', 'AUC (OvR)']):
    fig.add_hline(y=orig_metrics[metric], line_dash="dash", line_color="red",
                  annotation_text="Original Baseline", facet_col=i+1)

fig.write_image("synthetic_comparison.svg", width=1200, height=600)
fig.show()

fig_dist = px.bar(pd.DataFrame(dist_data), x="Dataset", y="Percentage", color="Class",
                  barmode="group", title="Class Ratio Fidelity (Original vs Synthetic)")
fig_dist.write_image("class_distribution_fidelity.svg", width=1200, height=600)
fig_dist.show()